<a href="https://colab.research.google.com/github/jdansb/Econophysics/blob/main/linear.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [91]:
pip install -q linearmodels

# OLS

In [117]:
# @title
import numpy as np
import statsmodels.api as sm


#Manual
I = np.random.randint(0,100)
x = np.random.randint(0, 100, I)
y = np.random.randint(0, 100, I)

#x = np.array([1,2,3,4,5,6,7,8,9,10])
#y = np.array([5,8,9,13,15,18,20,25,27,30])

N=len(x)
xm=np.mean(x)
ym=np.mean(y)
s1=0;s2=0
for i in (range(N)):
    s1+=x[i]*y[i]
    s2+=x[i]*x[i]

b1=(s1-N*ym*xm)/(s2-N*xm*xm)
b0=ym-b1*xm
print("Manual")
print("const: "+str(b0)+"\nx1:    "+str(b1))

### Auto


X = sm.add_constant(x)

# regressão OLS
modelo = sm.OLS(y, X).fit()

# resultados
# print(modelo.summary())
print("\nAuto")
print("const: "+str(modelo.params[0])+"\nx1:    "+str(modelo.params[1]))

print("\nOs coeficientes angulares são {:.2f}% iguais.".format(100*b1/modelo.params[1]))


Manual
const: 48.71590353827139
x1:    -0.06073409789888189

Auto
const: 48.7159035382714
x1:    -0.06073409789888215

Os coeficientes angulares são 100.00% iguais.


#OLS com transformação da média (FE)

In [118]:
# @title
from linearmodels.panel import PanelOLS
import pandas as pd


#Manual
x1 = np.random.randint(0, 100, I)
y1 = np.random.randint(0, 100, I)

nx=x-np.mean(x)
nx1=x1-np.mean(x1)
ny=y-np.mean(y)
ny1=y1-np.mean(y1)

x2=np.concatenate((nx,nx1))
y2=np.concatenate((ny,ny1))

N=len(x2)
xm=np.mean(x2)
ym=np.mean(y2)
s1=0;s2=0
for i in (range(N)):
    s1+=x2[i]*y2[i]
    s2+=x2[i]*x2[i]

b1=(s1-N*ym*xm)/(s2-N*xm*xm)
b0=ym-b1*xm
print("Manual")
print("x1:    "+str(b1))

#Auto
# dataframe painel
df = pd.DataFrame({
    'id': [1]*I + [2]*I,
    'tempo': list(range(I)) + list(range(I)),
    'x': np.concatenate([x, x1]),
    'y': np.concatenate([y, y1])
})

#print(df)

# índice painel
df = df.set_index(['id', 'tempo'])

# modelo FE
modelo = PanelOLS.from_formula(
    'y ~ x + EntityEffects',
    data=df
)

resultado = modelo.fit()

print("\nAuto")
print("x1:    "+str(resultado.params['x']))

print("\nOs coeficientes angulares são {:.2f}% iguais.".format(100*b1/resultado.params['x']))



Manual
x1:    0.1577693520786498

Auto
x1:    0.15776935207864973

Os coeficientes angulares são 100.00% iguais.


# OLS com transformação da média e ponderado (FE+Weighted)

In [119]:
# @title
M=len(nx) #Tamanho original de cada série
N=len(y2)

s1=0;s2=0;s3=0;s4=0;s5=0
for i in (range(N)):
    w = 0.25 if (i<M) else 0.75
    s1+=w*x2[i]*y2[i]
    s2+=w*y2[i]
    s3+=w*x2[i]
    s4+=w*x2[i]*x2[i]
    s5+=w*x2[i]

b1=(s1-s2*s3/M)/(s4-s5*s5/M)
print("Manual")
print("x1:    "+str(b1))

#Auto
df = pd.DataFrame({
    'id': [1]*I + [2]*I,
    'tempo': list(range(I)) + list(range(I)),
    'x': np.concatenate([x, x1]),
    'y': np.concatenate([y, y1])
})

df['peso'] = np.where(df['id'] == 1, 0.25, 0.75)

df = df.set_index(['id', 'tempo'])

modelo = PanelOLS.from_formula(
    'y ~ x + EntityEffects',
    data=df,
    weights=df['peso']
)

resultado = modelo.fit()

print("\nAuto")
print("x1:    "+str(resultado.params['x']))

print("\nOs coeficientes angulares são {:.2f}% iguais.".format(100*b1/resultado.params['x']))

Manual
x1:    0.2969533853308433

Auto
x1:    0.29695338533084326

Os coeficientes angulares são 100.00% iguais.


# Pearson

In [124]:
# @title
from scipy.stats import pearsonr


I = np.random.randint(0,100)
x = np.random.randint(0, 100, I)
y = np.random.randint(0, 100, I)

xn=x-np.mean(x)
yn=y-np.mean(y)
p1 = np.dot(xn, yn)
p2 = np.dot(xn, xn)
p3 = np.dot(yn, yn)
c=p1/(np.sqrt(p2)*np.sqrt(p3))
print("Manual")
print("Correlaçao:    "+str(c))

corr, _= pearsonr(x, y)

print("\nAuto")
print("Correlação:    "+str(corr))

print("\nOs coeficientes de correlação são {:.2f}% iguais.".format(100*c/corr))

Manual
Correlaçao:    0.012785408476462182

Auto
Correlação:    0.012785408476462127

Os coeficientes de correlação são 100.00% iguais.


Spearmn

In [123]:
# @title
from scipy.stats import rankdata
rx = rankdata(x)
ry = rankdata(y)

xn=rx-np.mean(rx)
yn=ry-np.mean(ry)
p1 = np.dot(xn, yn)
p2 = np.dot(xn, xn)
p3 = np.dot(yn, yn)
c=p1/(np.sqrt(p2)*np.sqrt(p3))
print("Manual")
print("Correlaçao:    "+str(c))

corr, _= spearmanr(x, y)

print("\nAuto")
print("Correlação:    "+str(corr))

print("\nOs coeficientes de correlação são {:.2f}% iguais.".format(100*c/corr))


Manual
Correlaçao:    0.07954084943381937

Auto
Correlação:    0.07954084943381935

Os coeficientes de correlação são 100.00% iguais.
